**ETL Pipeline in Detail**
1. Extract:
    * Extract data from the raw CSV files into pandas DataFrames.Read files like orders.csv,  products.csv, and order_products__train.csv and convert them into structured DataFrames.

2. Transform:
    * Impute Missing Values: Numerical columns with missing data are filled using an imputer (mean strategy).
    * Encode Categorical Data
    * Merge Datasets: Use pandas’ merge() function to combine the datasets on common columns (order_id and product_id).
    * Group Data: The order_id and product_id are grouped to create a list of products for each order, with products shown as a space-separated string.
    * Apply Transformers: Use transformers like GroupByOrderId to handle the grouping and transformation of data in a pipeline.

3. Load:
    * Instead of saving the result to a file, the grouped and transformed data is displayed within the notebook for review or further use.

**Benefits of ETL in This Process**
1. Data Integration: Data from different sources (orders, products, aisles, etc.) is combined into a unified format, making it easier to work with.
2. Data Cleaning: Missing values are handled, categorical data is encoded, and unnecessary columns are removed to ensure the data is ready for analysis or modeling.
3. Automation: The pipeline automates the process of loading, transforming, and grouping data, ensuring consistent and repeatable results without manual intervention.

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from IPython.display import display

# Step 1: Load the datasets
def load_datasets():
    aisles = pd.read_csv('/kaggle/input/supermarket-superstore-dataset-bundle/data/aisles.csv', delimiter=';')
    departments = pd.read_csv('/kaggle/input/supermarket-superstore-dataset-bundle/data/departments.csv', delimiter=';')
    order_products_train = pd.read_csv('/kaggle/input/supermarket-superstore-dataset-bundle/data/order_products__train.csv', delimiter=';')
    orders = pd.read_csv('/kaggle/input/supermarket-superstore-dataset-bundle/data/orders.csv', delimiter=';')
    products = pd.read_csv('/kaggle/input/supermarket-superstore-dataset-bundle/data/products.csv', delimiter=';')
    
    return aisles, departments, order_products_train, orders, products

# Step 2: Merge the datasets
def merge_datasets(order_products_train, orders, products):
    merged_data = pd.merge(order_products_train, orders, on='order_id', how='inner')
    merged_data = pd.merge(merged_data, products, on='product_id', how='inner')
    
    # Drop the 'Unnamed: 4' column
    merged_data = merged_data.drop(columns='Unnamed: 4')
    
    return merged_data

# Step 3: Preprocessing (Missing Values, Encoding)
class GroupByOrderId(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Grouping by 'order_id' and aggregating the 'product_id' into a space-separated string
        grouped_data = X.groupby('order_id')['product_id'].apply(lambda x: ' '.join(map(str, x))).reset_index()
        return grouped_data

def preprocess_data(merged_data):
    # Step 3.1: Impute missing values for numerical columns
    imputer = SimpleImputer(strategy='mean')
    merged_data_imputed = pd.DataFrame(imputer.fit_transform(merged_data.select_dtypes(include=['float64', 'int64'])))
    merged_data_imputed.columns = merged_data.select_dtypes(include=['float64', 'int64']).columns

    # Step 3.2: Encode categorical columns (e.g., 'eval_set')
    label_encoder = LabelEncoder()
    merged_data['eval_set'] = label_encoder.fit_transform(merged_data['eval_set'])
    
    # Step 3.3: Apply GroupByOrderId Transformer
    group_by_order_id = GroupByOrderId()
    grouped_data = group_by_order_id.fit_transform(merged_data)
    
    return grouped_data

# Step 4: Display the transformed data as output
def display_submission_data(grouped_data):
    # Rename columns as per the required output
    grouped_data.columns = ['order_id', 'products']
    
    # Display the grouped data
    display(grouped_data.head())  # Display the first few rows of the data
    
    return grouped_data

# Main ETL Pipeline Execution
if __name__ == "__main__":
    # Load datasets
    aisles, departments, order_products_train, orders, products = load_datasets()

    # Merge datasets
    merged_data = merge_datasets(order_products_train, orders, products)

    # Preprocess data (impute missing values, encode categorical columns, group data)
    grouped_data = preprocess_data(merged_data)

    # Display the final result
    display_submission_data(grouped_data)


,order_id,products
0,38,11913 18159 4461 21616 23622 32433 28842 42625...
1,96,20574 30391 40706 25610 27966 24489 39275
2,98,27966 8859 19731 43654 13176 4357 37664 34065 ...
3,226,39275 28199 24852 29883 28427 7754 39947 47307...
4,762,30391 21137 41220 15872
